# Synthetic Student Performance Dataset

This notebook checks the parent workspace for an existing CSV with the same dataset schema and at least 5,000 rows. If none is found, it creates a deterministic, fully fictitious dataset for simple linear regression (`exam_score` vs. `study_hours_per_week`), multiple linear regression, and logistic regression (`passed`).

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

N_ROWS = 5_000
RANDOM_SEED = 42
DATASET_NAME = "synthetic_student_performance.csv"
NOTEBOOK_DIR = Path.cwd()
SEARCH_ROOT = NOTEBOOK_DIR.parent
REQUIRED_COLUMNS = {
    "student_id",
    "study_hours_per_week",
    "attendance_rate_pct",
    "previous_exam_score",
    "sleep_hours_per_night",
    "assignment_completion_pct",
    "tutoring_hours_per_month",
    "class_participation_score",
    "internet_access",
    "gender",
    "parent_education_level",
    "exam_score",
    "passed",
}


def is_similar_dataset(csv_path):
    try:
        columns = set(pd.read_csv(csv_path, nrows=0).columns)
        if not REQUIRED_COLUMNS.issubset(columns):
            return False

        row_count = 0
        for chunk in pd.read_csv(
            csv_path, usecols=sorted(REQUIRED_COLUMNS), chunksize=1_000
        ):
            row_count += len(chunk)
            if row_count >= N_ROWS:
                return True
        return False
    except (OSError, UnicodeDecodeError, pd.errors.EmptyDataError, pd.errors.ParserError, ValueError) as error:
        print(f"Could not check {csv_path}: {error}")
        return False


csv_paths = []
for folder, subfolders, filenames in os.walk(SEARCH_ROOT):
    subfolders[:] = [
        name for name in subfolders
        if name.lower() not in {".venv", ".git", "__pycache__"}
    ]
    csv_paths.extend(
        Path(folder) / filename
        for filename in filenames
        if filename.lower().endswith(".csv")
    )

existing_dataset = next(
    (path for path in sorted(csv_paths) if is_similar_dataset(path)),
    None,
)

if existing_dataset is not None:
    DATASET_PATH = existing_dataset
    print(f"Using existing similar dataset; no new CSV created: {DATASET_PATH}")
else:
    rng = np.random.default_rng(RANDOM_SEED)

    study_hours = np.clip(rng.normal(12, 5, N_ROWS), 0, 35).round(1)
    attendance = np.clip(rng.normal(82, 10, N_ROWS), 45, 100).round(1)
    previous_score = np.clip(rng.normal(68, 12, N_ROWS), 0, 100).round(1)
    sleep_hours = np.clip(rng.normal(7, 1, N_ROWS), 4, 10).round(1)
    assignment_completion = np.clip(rng.normal(78, 15, N_ROWS), 20, 100).round(1)
    tutoring_hours = np.clip(rng.poisson(3, N_ROWS), 0, 15)
    participation = np.clip(rng.normal(5, 2, N_ROWS), 0, 10).round(1)
    internet_access = rng.choice(["yes", "no"], N_ROWS, p=[0.88, 0.12])
    gender = rng.choice(["female", "male", "non_binary"], N_ROWS, p=[0.49, 0.49, 0.02])
    parent_education = rng.choice(
        ["high_school", "some_college", "bachelors", "graduate"],
        N_ROWS,
        p=[0.30, 0.28, 0.27, 0.15],
    )

    exam_score = (
        65
        + 0.85 * (study_hours - 12)
        + 0.20 * (attendance - 82)
        + 0.22 * (previous_score - 68)
        + 0.12 * (assignment_completion - 78)
        + 0.65 * (tutoring_hours - 3)
        + 0.80 * (participation - 5)
        + 1.10 * (sleep_hours - 7)
        + rng.normal(0, 8, N_ROWS)
    )
    exam_score = np.clip(exam_score, 0, 100).round(1)

    pass_logit = (
        -0.15
        + 0.14 * (study_hours - 12)
        + 0.045 * (attendance - 82)
        + 0.05 * (previous_score - 68)
        + 0.035 * (assignment_completion - 78)
        + 0.17 * (tutoring_hours - 3)
        + 0.12 * (participation - 5)
        + 0.15 * (sleep_hours - 7)
        + 0.30 * (internet_access == "yes")
    )
    pass_probability = 1 / (1 + np.exp(-pass_logit))
    passed = rng.binomial(1, pass_probability)

    student_data = pd.DataFrame(
        {
            "student_id": [f"S{index:05d}" for index in range(1, N_ROWS + 1)],
            "study_hours_per_week": study_hours,
            "attendance_rate_pct": attendance,
            "previous_exam_score": previous_score,
            "sleep_hours_per_night": sleep_hours,
            "assignment_completion_pct": assignment_completion,
            "tutoring_hours_per_month": tutoring_hours,
            "class_participation_score": participation,
            "internet_access": internet_access,
            "gender": gender,
            "parent_education_level": parent_education,
            "exam_score": exam_score,
            "passed": passed,
        }
    )

    DATASET_PATH = NOTEBOOK_DIR / DATASET_NAME
    student_data.to_csv(DATASET_PATH, index=False)
    print(f"Created {len(student_data):,} fictitious observations: {DATASET_PATH}")

student_data = pd.read_csv(DATASET_PATH)
print(f"Dataset shape: {student_data.shape}")
display(student_data.head())
display(student_data[["exam_score", "passed"]].describe())
print("Pass/fail counts:")
display(student_data["passed"].value_counts().sort_index().rename(index={0: "fail", 1: "pass"}))